# Data Preprocessing and Feature Engineering Pipeline

## Project

Linear and Regularized Regression for Used-Car Price Prediction

## Notebook Purpose

This notebook converts the audited raw vehicle dataset into a reproducible and
model-ready feature set.

The preprocessing decisions are based on the findings of the previous data
understanding and exploratory data analysis notebooks.

The main objectives are:

- separate input features and the target variable;
- split the data before fitting learned preprocessing operations;
- create justified numerical features from raw vehicle specifications;
- handle missing numerical and categorical values;
- standardize numerical features;
- encode categorical variables;
- control rare and high-cardinality categories;
- combine all preprocessing steps into a reusable pipeline;
- verify the final processed feature matrix.

No regression model is trained in this notebook. The fitted preprocessing
pipeline will later be used consistently by Linear Regression, Ridge, Lasso,
and Elastic Net models.

## 1. Preprocessing Principles

The raw CSV file will remain unchanged. All feature engineering and
preprocessing operations will be performed on working copies or inside
scikit-learn pipelines.

The dataset will be split before fitting imputation, scaling, and categorical
encoding. This prevents information from the test set from influencing the
training process.

The same preprocessing protocol and train-test split will be used for all
regression models to ensure a fair comparison.

## 2. Import Required Libraries

This notebook uses Pandas and NumPy for data manipulation and feature
engineering. Scikit-learn utilities are used for data splitting, missing-value
imputation, numerical scaling, categorical encoding, and pipeline construction.

Regression algorithms are not imported because model training is outside the
scope of this notebook.

In [1]:
import re

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3. Load and Validate the Raw Dataset

The raw CSV file is loaded independently so that this notebook can run without
executing the previous notebooks.

A small schema validation confirms that the target and essential raw features
are present. The original data is stored in `raw_df` and will not be modified
directly.

In [2]:
DATA_PATH = "../data/raw/car_price_data.csv"
TARGET_COLUMN = "Price"

required_columns = [
    "Price",
    "Year",
    "Kilometer",
    "Engine",
    "Max Power",
    "Max Torque"
]

raw_df = pd.read_csv(DATA_PATH)

missing_columns = [
    column
    for column in required_columns
    if column not in raw_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

if raw_df[TARGET_COLUMN].isna().any():
    raise ValueError("The target column contains missing values.")

print("Dataset loaded and validated successfully.")
print("Dataset shape:", raw_df.shape)
print("Target column:", TARGET_COLUMN)
print("Missing target values:", raw_df[TARGET_COLUMN].isna().sum())

raw_df.head()

Dataset loaded and validated successfully.
Dataset shape: (2059, 20)
Target column: Price
Missing target values: 0


,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0


## 4. Separate Input Features and Target

The target variable must be separated from the input features before data
splitting and preprocessing.

`X_raw` contains the original candidate input features, while `y` contains the
vehicle Price that the regression models will learn to predict.

Price must not remain inside the input features because this would cause direct
target leakage.

In [3]:
X_raw = raw_df.drop(
    columns=[TARGET_COLUMN]
).copy()

y = raw_df[TARGET_COLUMN].copy()

print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)
print("Target inside X_raw:", TARGET_COLUMN in X_raw.columns)

X_raw shape: (2059, 19)
y shape: (2059,)
Target inside X_raw: False


## 5. Create an Honest Train-Test Split

The dataset is divided into training and test sets before fitting any
data-dependent preprocessing operation.

The training set will be used to learn imputation values, scaling parameters,
categorical levels, and regression coefficients. The test set will remain
unseen until final model evaluation.

A fixed random state makes the split reproducible. The same split will be used
for Linear Regression, Ridge, Lasso, and Elastic Net to ensure a fair
comparison.

In [4]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

(
    X_train_raw,
    X_test_raw,
    y_train,
    y_test
) = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

shared_indices = set(
    X_train_raw.index
).intersection(
    X_test_raw.index
)

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Shared row indices:", len(shared_indices))

X_train_raw shape: (1647, 19)
X_test_raw shape: (412, 19)
y_train shape: (1647,)
y_test shape: (412,)
Shared row indices: 0


### 5.1 Validate the Target Distribution After Splitting

A random split can place very expensive or unusual vehicles unevenly between
the training and test sets.

The target summaries are compared to confirm that both sets represent the main
Price distribution reasonably well. The test set does not need to be identical
to the training set, but severe differences could make evaluation unreliable.

In [5]:
target_split_summary = pd.DataFrame({
    "Training Target": y_train.describe(),
    "Test Target": y_test.describe()
})

target_split_summary.loc["skewness"] = [
    y_train.skew(),
    y_test.skew()
]

target_split_summary.round(2)

,Training Target,Test Target
count,1647.00,412.00
mean,1696655.08,1728322.77
std,2360648.88,2646371.18
min,49000.00,130000.00
25%,477500.00,495000.00
50%,825000.00,837500.00
75%,1950000.00,1850000.00
max,27500000.00,35000000.00
skewness,4.45,6.38


### Split-Validation Interpretation

The training and test sets have similar means, medians, and interquartile
ranges. Therefore, both sets reasonably represent the main vehicle Price
distribution.

The test set has a higher standard deviation and skewness because the
35-million maximum-price vehicle is included in the test set, while the
maximum training Price is 27.5 million.

The split will not be manually changed because moving extreme observations
between sets would introduce selection bias. The expensive test vehicle will
provide an honest extrapolation challenge.

MAE, RMSE, and R² will be reported together because RMSE may be strongly
influenced by this extreme test observation. Cross-validation and sensitivity
analysis will later be used to assess the stability of model performance.

## 6. Prepare Original and Log-Transformed Targets

EDA found that Price is strongly right-skewed. Therefore, two target
representations will be retained:

- original Price for predictions and errors in currency units;
- log-transformed Price for reducing the influence of extremely expensive
  vehicles and modeling relative Price differences.

The transformation is applied separately after splitting. Both original
targets are preserved for evaluation on the real Price scale.

In [6]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(
    "Training Price skewness:",
    round(y_train.skew(), 2)
)

print(
    "Training Log Price skewness:",
    round(y_train_log.skew(), 2)
)

print(
    "Test Price skewness:",
    round(y_test.skew(), 2)
)

print(
    "Test Log Price skewness:",
    round(y_test_log.skew(), 2)
)

Training Price skewness: 4.45
Training Log Price skewness: 0.46
Test Price skewness: 6.38
Test Log Price skewness: 0.58


### Target-Transformation Interpretation

The log transformation substantially reduced the right skewness of Price in
both data subsets.

Training skewness decreased from 4.45 to 0.46, while test skewness decreased
from 6.38 to 0.58. This makes the target distributions more balanced and
reduces the relative influence of extremely expensive vehicles.

This does not prove perfect normality or guarantee better model performance.
Models using original and log-transformed targets will therefore be compared
through cross-validation, test metrics, and residual diagnostics.

## 7. Deterministic Feature Engineering

Several technical specifications are stored as text even though they contain
useful numerical information.

Deterministic feature engineering converts these strings into numerical
features using fixed rules. These operations do not calculate statistics from
the training or test data and therefore do not learn from the test set.

The raw DataFrames will remain unchanged.

### 7.1 Define Numerical Extraction Functions

def extract_first_number(text_series):
    extracted_number = text_series.astype(
        "string"
    ).str.extract(
        r"(\d+(?:\.\d+)?)",
        expand=False
    )

    return pd.to_numeric(
        extracted_number,
        errors="coerce"
    ).astype(float)


def extract_rpm(text_series):
    extracted_rpm = text_series.astype(
        "string"
    ).str.extract(
        r"@\s*(\d+(?:\.\d+)?)",
        expand=False
    )

    return pd.to_numeric(
        extracted_rpm,
        errors="coerce"
    ).astype(float)

### 7.2 Validate Numerical Extraction

The extraction functions are tested on a small sample before being applied to
the complete training and test sets.

The raw text and extracted numerical values are displayed together so that
incorrect parsing can be detected visually.

In [9]:
extraction_demo = X_train_raw[
    [
        "Engine",
        "Max Power",
        "Max Torque"
    ]
].head(10).copy()

extraction_demo["Engine_CC"] = extract_first_number(
    extraction_demo["Engine"]
)

extraction_demo["Power_BHP"] = extract_first_number(
    extraction_demo["Max Power"]
)

extraction_demo["Power_RPM"] = extract_rpm(
    extraction_demo["Max Power"]
)

extraction_demo["Torque_NM"] = extract_first_number(
    extraction_demo["Max Torque"]
)

extraction_demo["Torque_RPM"] = extract_rpm(
    extraction_demo["Max Torque"]
)

extraction_demo

,Engine,Max Power,Max Torque,Engine_CC,Power_BHP,Power_RPM,Torque_NM,Torque_RPM
266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1133,1995 cc,184 bhp @ 4000 rpm,350 Nm @ 1750 rpm,1995.0,184.0,4000.0,350.0,1750.0
1823,1997 cc,138 bhp @ 3750 rpm,320 Nm @ 1600 rpm,1997.0,138.0,3750.0,320.0,1600.0
1370,2925 cc,326 bhp @ 3600 rpm,700 Nm @ 1200 rpm,2925.0,326.0,3600.0,700.0,1200.0
67,2755 cc,174 bhp @ 3400 rpm,450 Nm @ 1600 rpm,2755.0,174.0,3400.0,450.0,1600.0
1009,1248 cc,89 bhp @ 4000 rpm,200 Nm @ 1750 rpm,1248.0,89.0,4000.0,200.0,1750.0
1763,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,1248.0,74.0,4000.0,190.0,2000.0
1970,1950 cc,192 bhp @ 3800 rpm,400 Nm @ 1600 rpm,1950.0,192.0,3800.0,400.0,1600.0
1313,2993 cc,244 bhp @ 4000 rpm,600 Nm @ 2000 rpm,2993.0,244.0,4000.0,600.0,2000.0
1537,1498 cc,99 bhp @ 3750 rpm,205 Nm @ 1750 rpm,1498.0,99.0,3750.0,205.0,1750.0


### Extraction Validation Result

The extraction rules correctly converted the sampled Engine, Power, Torque,
and RPM strings into numerical values.

The first sampled observation had missing raw technical specifications, so its
derived features also remained missing. This is expected and confirms that the
functions do not invent unavailable information.

Missing derived values will later be handled by the training-fitted numerical
imputer.

In [10]:
REFERENCE_YEAR = 2022


def engineer_vehicle_features(data):
    engineered = data.copy()

    # Remove accidental spaces from text categories.
    text_columns = engineered.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        engineered[column] = (
            engineered[column]
            .str.strip()
        )

    # Convert manufacturing year into vehicle age.
    engineered["Vehicle Age"] = (
        REFERENCE_YEAR
        - pd.to_numeric(
            engineered["Year"],
            errors="coerce"
        )
    )

    # Extract numerical technical specifications.
    engineered["Engine_CC"] = extract_first_number(
        engineered["Engine"]
    )

    engineered["Power_BHP"] = extract_first_number(
        engineered["Max Power"]
    )

    engineered["Power_RPM"] = extract_rpm(
        engineered["Max Power"]
    )

    engineered["Torque_NM"] = extract_first_number(
        engineered["Max Torque"]
    )

    engineered["Torque_RPM_Clean"] = extract_rpm(
        engineered["Max Torque"]
    )

    # Convert the suspicious 150-RPM value into missing.
    engineered.loc[
        engineered["Torque_RPM_Clean"] < 500,
        "Torque_RPM_Clean"
    ] = np.nan

    # Standardize logically equivalent category labels.
    engineered["Fuel Type"] = (
        engineered["Fuel Type"].replace({
            "CNG + CNG": "CNG"
        })
    )

    engineered["Owner"] = (
        engineered["Owner"].replace({
            "Fourth": "Fourth or More",
            "4 or More": "Fourth or More"
        })
    )

    # Remove raw columns replaced by engineered features.
    engineered = engineered.drop(
        columns=[
            "Year",
            "Engine",
            "Max Power",
            "Max Torque"
        ]
    )

    return engineered

### 7.4 Apply and Validate Feature Engineering

The same deterministic feature-engineering function is applied independently
to the training and test feature sets.

The resulting shapes, column changes, and missing derived values are examined
before constructing the learned preprocessing pipeline.

In [11]:
X_train_engineered = engineer_vehicle_features(
    X_train_raw
)

X_test_engineered = engineer_vehicle_features(
    X_test_raw
)

engineered_features = [
    "Vehicle Age",
    "Engine_CC",
    "Power_BHP",
    "Power_RPM",
    "Torque_NM",
    "Torque_RPM_Clean"
]

engineered_missing_summary = pd.DataFrame({
    "Training Missing": (
        X_train_engineered[
            engineered_features
        ].isna().sum()
    ),
    "Test Missing": (
        X_test_engineered[
            engineered_features
        ].isna().sum()
    )
})

print(
    "Engineered training shape:",
    X_train_engineered.shape
)

print(
    "Engineered test shape:",
    X_test_engineered.shape
)

print(
    "Year in raw training data:",
    "Year" in X_train_raw.columns
)

print(
    "Year in engineered training data:",
    "Year" in X_train_engineered.columns
)

print("\nMissing engineered values:")

engineered_missing_summary

Engineered training shape: (1647, 21)
Engineered test shape: (412, 21)
Year in raw training data: True
Year in engineered training data: False

Missing engineered values:


,Training Missing,Test Missing
Vehicle Age,0,0
Engine_CC,65,15
Power_BHP,65,15
Power_RPM,69,15
Torque_NM,65,15
Torque_RPM_Clean,65,16


### Feature-Engineering Validation Result

The engineered training and test sets contain 21 features each. The raw
training data still contains Year, confirming that the feature-engineering
function did not modify the original DataFrame.

The 80 original missing technical records remain missing after extraction.
Power_RPM has four additional missing training values because the corresponding
raw strings did not provide an RPM number.

Torque_RPM_Clean has one additional missing test value because the suspicious
150-RPM observation was converted to missing rather than corrected through
guessing.

No rows were deleted. These missing numerical values will later be imputed
using medians learned from the training data only.

## 8. Define Feature Groups

Numerical and categorical features require different preprocessing operations.

Numerical features will receive median imputation and standardization.
Categorical features will receive explicit missing-value treatment and
one-hot encoding.

Feature lists are defined explicitly so that the preprocessing design remains
transparent and reproducible.

In [12]:
numerical_features = [
    "Kilometer",
    "Length",
    "Width",
    "Height",
    "Seating Capacity",
    "Fuel Tank Capacity",
    "Vehicle Age",
    "Engine_CC",
    "Power_BHP",
    "Power_RPM",
    "Torque_NM",
    "Torque_RPM_Clean"
]

categorical_features = [
    "Make",
    "Model",
    "Fuel Type",
    "Transmission",
    "Location",
    "Color",
    "Owner",
    "Seller Type",
    "Drivetrain"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 12
Categorical features: 9
Total features: 21


## 9. Numerical Preprocessing Pipeline

Missing numerical values will be replaced with training-set medians.
Missing-value indicators will preserve information about which values were
originally unavailable.

After imputation, numerical features will be standardized for fair
regularization.

In [13]:
numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

## 10. Categorical Preprocessing Pipeline

Missing categorical values will be represented by an explicit `Missing`
category.

Categorical values will then be converted into numerical binary columns.
Categories appearing fewer than five times in the training data will be grouped
to reduce sparse and unstable features.

In [21]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5,
            drop="first",
            sparse_output=False
        )
    )
])

## 11. Combine Numerical and Categorical Preprocessing

The ColumnTransformer sends numerical features through the numerical pipeline
and categorical features through the categorical pipeline.

This ensures that each feature type receives the correct preprocessing.

In [22]:
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

## 12. Fit and Apply the Preprocessing Pipeline

The preprocessor is fitted only on the training data. It learns training-set
medians, means, standard deviations, categories, and rare-category groups.

The fitted preprocessor is then applied to the test data without learning
anything from it.

In [23]:
X_train_processed = preprocessor.fit_transform(
    X_train_engineered
)

X_test_processed = preprocessor.transform(
    X_test_engineered
)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (1647, 160)
Processed test shape: (412, 160)


C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


### Processed-Matrix Result

The training and test sets were successfully transformed into 169 model-ready
features.

Both sets contain the same number and order of processed columns. The increase
from 21 engineered features to 169 processed features is mainly caused by
one-hot encoding and numerical missing-value indicators.

The different row counts preserve the original 80/20 train-test split.

In [24]:
print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Test missing values:",
    np.isnan(X_test_processed).sum()
)

print(
    "Training infinite values:",
    np.isinf(X_train_processed).sum()
)

print(
    "Test infinite values:",
    np.isinf(X_test_processed).sum()
)

print(
    "Same number of columns:",
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

Training missing values: 0
Test missing values: 0
Training infinite values: 0
Test infinite values: 0
Same number of columns: True


### Preprocessing Validation Result

The processed training and test matrices contain no missing or infinite values.

Both matrices have the same number of columns, confirming that the fitted
training preprocessing structure was applied consistently to the test data.

The processed features are now technically suitable for regression modeling.

## 13. Inspect Processed Feature Names

The processed feature names are retained for coefficient interpretation,
feature comparison, and model diagnostics in later notebooks.

In [25]:
feature_names = preprocessor.get_feature_names_out()

print("Number of feature names:", len(feature_names))
print("\nFirst 20 feature names:")

feature_names[:20]

Number of feature names: 160

First 20 feature names:


array(['numerical__Kilometer', 'numerical__Length', 'numerical__Width',
       'numerical__Height', 'numerical__Seating Capacity',
       'numerical__Fuel Tank Capacity', 'numerical__Vehicle Age',
       'numerical__Engine_CC', 'numerical__Power_BHP',
       'numerical__Power_RPM', 'numerical__Torque_NM',
       'numerical__Torque_RPM_Clean',
       'numerical__missingindicator_Length',
       'numerical__missingindicator_Width',
       'numerical__missingindicator_Height',
       'numerical__missingindicator_Seating Capacity',
       'numerical__missingindicator_Fuel Tank Capacity',
       'numerical__missingindicator_Engine_CC',
       'numerical__missingindicator_Power_BHP',
       'numerical__missingindicator_Power_RPM'], dtype=object)

### Feature-Name Interpretation

The number of extracted feature names matches the 169 columns in the processed
matrices.

The first 12 columns are standardized numerical features. The
`missingindicator` columns record whether selected numerical values were
originally missing before median imputation.

The remaining columns are one-hot-encoded categorical features. These names
will later allow regression coefficients to be linked to understandable
vehicle characteristics.

In [26]:
categorical_names = [
    name
    for name in feature_names
    if name.startswith("categorical__")
]

print(
    "Categorical encoded features:",
    len(categorical_names)
)

print("\nFirst 20 categorical features:")

categorical_names[:20]

Categorical encoded features: 138

First 20 categorical features:


['categorical__Make_BMW',
 'categorical__Make_Chevrolet',
 'categorical__Make_Datsun',
 'categorical__Make_Ford',
 'categorical__Make_Honda',
 'categorical__Make_Hyundai',
 'categorical__Make_Jaguar',
 'categorical__Make_Jeep',
 'categorical__Make_Kia',
 'categorical__Make_Land Rover',
 'categorical__Make_MG',
 'categorical__Make_MINI',
 'categorical__Make_Mahindra',
 'categorical__Make_Maruti Suzuki',
 'categorical__Make_Mercedes-Benz',
 'categorical__Make_Nissan',
 'categorical__Make_Porsche',
 'categorical__Make_Renault',
 'categorical__Make_Skoda',
 'categorical__Make_Tata']

### 13.1 Encoded Columns by Categorical Feature

The number of generated columns is examined for each categorical feature.
This helps verify the effect of rare-category grouping, particularly for the
high-cardinality Model feature.

In [27]:
for column in categorical_features:
    count = sum(
        name.startswith(
            f"categorical__{column}_"
        )
        for name in categorical_names
    )

    print(column, ":", count)

print("\nInfrequent-category columns:")

[
    name
    for name in categorical_names
    if "infrequent" in name
]

Make : 24
Model : 46
Fuel Type : 4
Transmission : 1
Location : 39
Color : 15
Owner : 4
Seller Type : 2
Drivetrain : 3

Infrequent-category columns:


['categorical__Make_infrequent_sklearn',
 'categorical__Model_infrequent_sklearn',
 'categorical__Fuel Type_infrequent_sklearn',
 'categorical__Location_infrequent_sklearn',
 'categorical__Color_infrequent_sklearn',
 'categorical__Owner_infrequent_sklearn']

In [28]:
print("Training shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)
print("Feature names:", len(feature_names))
print("Categorical features:", len(categorical_names))

Training shape: (1647, 160)
Test shape: (412, 160)
Feature names: 160
Categorical features: 138


In [29]:
print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Test missing values:",
    np.isnan(X_test_processed).sum()
)

print(
    "Training infinite values:",
    np.isinf(X_train_processed).sum()
)

print(
    "Test infinite values:",
    np.isinf(X_test_processed).sum()
)

Training missing values: 0
Test missing values: 0
Training infinite values: 0
Test infinite values: 0


## 14. Notebook Conclusion

This notebook developed and validated a leakage-safe preprocessing workflow for
used-car Price prediction.

The main completed steps were:

- loaded and validated the raw dataset;
- separated input features and the Price target;
- created a reproducible 80/20 train-test split;
- retained both original and log-transformed target representations;
- created Vehicle Age from manufacturing Year;
- extracted Engine_CC, Power_BHP, Power_RPM, Torque_NM, and
  Torque_RPM_Clean from raw specification strings;
- retained unavailable technical information as missing rather than guessing;
- handled the suspicious 150-RPM value without modifying the raw CSV;
- standardized logically equivalent category labels;
- divided the engineered features into numerical and categorical groups;
- used median imputation and missing-value indicators for numerical features;
- standardized numerical features;
- represented missing categorical values explicitly;
- grouped categories occurring fewer than five times;
- applied one-hot encoding with a reference category;
- fitted all learned preprocessing operations on training data only;
- transformed the training and test sets into 160 consistent model-ready
  features;
- confirmed that the processed matrices contain no missing or infinite values.

The original raw dataset remains unchanged.

The next notebook will use this preprocessing design with Linear Regression.
Preprocessing and the regression model will be combined into one pipeline so
that cross-validation learns imputation, scaling, and encoding independently
within each training fold.

The high-cardinality Model feature will later be evaluated through controlled
experiments with and without Model. Model performance will be assessed using
MAE, RMSE, R², residual diagnostics, and sensitivity to extreme vehicles.